# 🛡️ DeepGuard AI — Deep Learning Fraud Detection

**Capstone Project · Deep Learning with PyTorch**

This notebook trains a PyTorch feed-forward neural network to flag potentially fraudulent credit-card transactions.
It mirrors `app/train.py` from the DeepGuard AI project, adding charts so every step can be explained.

**Workflow**
1. Load and validate the data
2. Explore the class imbalance
3. Split 70 / 15 / 15 (stratified) and scale on training data only
4. Define and train `FraudNet` with a weighted loss
5. Choose the alert threshold on the validation set
6. Evaluate once on the held-out test set
7. Score individual transactions and save the artifacts

**Dataset:** [Credit Card Fraud Detection — mlg-ulb/creditcardfraud](https://www.kaggle.com/datasets/mlg-ulb/creditcardfraud/data)
(284,807 transactions by European cardholders, September 2013; 492 frauds).

**Kaggle setup:** *Add Input → Datasets →* search `creditcardfraud` (by mlg-ulb). If it is not attached, the notebook downloads it with `kagglehub` (requires internet access). A GPU accelerator is optional; the model is small and trains in a few minutes on CPU.

## 1. Setup

In [ ]:
import glob
import json
import random
from pathlib import Path

import joblib
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import torch
from sklearn.metrics import (
    ConfusionMatrixDisplay,
    average_precision_score,
    classification_report,
    confusion_matrix,
    precision_recall_curve,
    roc_auc_score,
    roc_curve,
)
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from torch import nn
from torch.utils.data import DataLoader, TensorDataset

SEED = 42
BATCH_SIZE = 2048
EPOCHS = 10
LEARNING_RATE = 0.001

random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)

if torch.cuda.is_available():
    device = torch.device("cuda")
elif torch.backends.mps.is_available():
    device = torch.device("mps")
else:
    device = torch.device("cpu")

print("PyTorch:", torch.__version__)
print("Device:", device)

## 2. Load and validate the data

Source: https://www.kaggle.com/datasets/mlg-ulb/creditcardfraud/data

In [ ]:
DATASET = "mlg-ulb/creditcardfraud"

# 1) Dataset attached on Kaggle, 2) local project copy, 3) download with kagglehub.
candidates = glob.glob("/kaggle/input/**/creditcard.csv", recursive=True) + ["data/raw/creditcard.csv"]
DATA_PATH = next((p for p in candidates if Path(p).exists()), None)
if DATA_PATH is None:
    try:
        import kagglehub
        DATA_PATH = str(Path(kagglehub.dataset_download(DATASET)) / "creditcard.csv")
    except Exception as exc:
        raise FileNotFoundError(
            f"creditcard.csv not found. On Kaggle: Add Input -> {DATASET}, "
            "or enable internet so kagglehub can download it."
        ) from exc

ARTIFACT_DIR = Path("/kaggle/working/artifacts") if Path("/kaggle/working").exists() else Path("artifacts_notebook")
ARTIFACT_DIR.mkdir(parents=True, exist_ok=True)

df = pd.read_csv(DATA_PATH)
print("Loaded:", DATA_PATH, df.shape)

if "Class" not in df.columns:
    raise ValueError("Dataset must contain the Class target column.")

feature_columns = [c for c in df.columns if c != "Class"]
if len(feature_columns) != 30:
    raise ValueError(f"Expected 30 input features, got {len(feature_columns)}.")

X = df[feature_columns].astype("float32").to_numpy()
y = df["Class"].astype("int64").to_numpy()

if not np.isfinite(X).all():
    raise ValueError("Features contain NaN or infinite values.")
if not set(np.unique(y)).issubset({0, 1}):
    raise ValueError("Class must contain only 0 and 1.")

df.head()

## 3. Explore the class imbalance

Fraud is extremely rare. A model that always predicts *legitimate* would be ~99.83% accurate and catch no fraud,
so accuracy is the wrong metric. We use **precision, recall, F1, PR-AUC and ROC-AUC** instead.

In [ ]:
counts = df["Class"].value_counts().sort_index()
print(f"Transactions: {len(df):,}")
print(f"Fraud cases:  {counts[1]:,}  ({counts[1] / len(df):.3%})")
print(f"Accuracy of always predicting 'legitimate': {counts[0] / len(df):.2%}")

fig, axes = plt.subplots(1, 2, figsize=(12, 4))
axes[0].bar(["Legitimate", "Fraud"], counts.values, color=["#0F766E", "#EE4C2C"])
axes[0].set_yscale("log")
axes[0].set_title("Class distribution (log scale)")
for i, v in enumerate(counts.values):
    axes[0].text(i, v, f"{v:,}", ha="center", va="bottom")

axes[1].hist(df.loc[df.Class == 0, "Amount"], bins=60, range=(0, 1000), alpha=0.6, density=True, label="Legitimate", color="#0F766E")
axes[1].hist(df.loc[df.Class == 1, "Amount"], bins=60, range=(0, 1000), alpha=0.6, density=True, label="Fraud", color="#EE4C2C")
axes[1].set_title("Transaction amount (0–1000)")
axes[1].legend()
plt.tight_layout()
plt.show()

## 4. Split and scale — without data leakage

- **Stratified** 70 / 15 / 15 split keeps the 0.17% fraud rate in every part.
- The `StandardScaler` learns mean and standard deviation from the **training set only**, then transforms validation and test.

In [ ]:
X_train, X_temp, y_train, y_temp = train_test_split(
    X, y, test_size=0.30, random_state=SEED, stratify=y
)
X_val, X_test, y_val, y_test = train_test_split(
    X_temp, y_temp, test_size=0.50, random_state=SEED, stratify=y_temp
)

scaler = StandardScaler()
X_train = scaler.fit_transform(X_train).astype("float32")
X_val = scaler.transform(X_val).astype("float32")
X_test = scaler.transform(X_test).astype("float32")

for name, labels in [("Train", y_train), ("Validation", y_val), ("Test", y_test)]:
    print(f"{name:<11} rows={len(labels):>7,}  fraud={int(labels.sum()):>3}  rate={labels.mean():.3%}")

## 5. The model: `FraudNet`

A feed-forward neural network (multilayer perceptron):

```
30 inputs → Linear 64 → ReLU → Dropout 0.2 → Linear 32 → ReLU → Dropout 0.1 → Linear 1 (logit)
```

The network outputs a raw **logit**; a sigmoid turns it into a 0–1 risk score at inference time.

In [ ]:
class FraudNet(nn.Module):
    """Small feed-forward neural network for binary fraud classification."""

    def __init__(self, input_size: int = 30):
        super().__init__()
        self.network = nn.Sequential(
            nn.Linear(input_size, 64),
            nn.ReLU(),
            nn.Dropout(0.20),
            nn.Linear(64, 32),
            nn.ReLU(),
            nn.Dropout(0.10),
            nn.Linear(32, 1),
        )

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        # Raw logits; BCEWithLogitsLoss applies the sigmoid internally.
        return self.network(x).squeeze(-1)


model = FraudNet(input_size=len(feature_columns)).to(device)
print(model)
print("Trainable weights:", sum(p.numel() for p in model.parameters() if p.requires_grad))

## 6. Train with a weighted loss

`pos_weight = negatives / positives` (≈ 579) makes missing a fraud cost far more than a false alarm.
Without it, the network learns to predict *legitimate* for everything.

In [ ]:
train_loader = DataLoader(
    TensorDataset(torch.from_numpy(X_train), torch.from_numpy(y_train.astype("float32"))),
    batch_size=BATCH_SIZE,
    shuffle=True,
)

positives = max(int(y_train.sum()), 1)
negatives = int(len(y_train) - y_train.sum())
pos_weight = torch.tensor([negatives / positives], dtype=torch.float32, device=device)
print(f"pos_weight = {pos_weight.item():.1f}")

criterion = nn.BCEWithLogitsLoss(pos_weight=pos_weight)
optimizer = torch.optim.Adam(model.parameters(), lr=LEARNING_RATE)


def scores_for(data):
    """Return sigmoid risk scores for a scaled feature matrix."""
    model.eval()
    out = []
    with torch.no_grad():
        for start in range(0, len(data), BATCH_SIZE):
            xb = torch.from_numpy(data[start:start + BATCH_SIZE]).to(device)
            out.append(torch.sigmoid(model(xb)).cpu().numpy())
    return np.concatenate(out)


history = {"loss": [], "val_pr_auc": []}
for epoch in range(EPOCHS):
    model.train()
    total_loss = 0.0
    for xb, yb in train_loader:
        xb, yb = xb.to(device), yb.to(device)
        optimizer.zero_grad()
        loss = criterion(model(xb), yb)
        loss.backward()
        optimizer.step()
        total_loss += loss.item() * len(xb)

    epoch_loss = total_loss / len(X_train)
    val_pr_auc = average_precision_score(y_val, scores_for(X_val))
    history["loss"].append(epoch_loss)
    history["val_pr_auc"].append(val_pr_auc)
    print(f"Epoch {epoch + 1:02d}/{EPOCHS} | loss={epoch_loss:.5f} | val PR-AUC={val_pr_auc:.4f}")

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 4))
epochs = range(1, EPOCHS + 1)
axes[0].plot(epochs, history["loss"], marker="o", color="#1E293B")
axes[0].set_title("Training loss (weighted BCE)")
axes[0].set_xlabel("Epoch")
axes[1].plot(epochs, history["val_pr_auc"], marker="o", color="#EE4C2C")
axes[1].set_title("Validation PR-AUC")
axes[1].set_xlabel("Epoch")
plt.tight_layout()
plt.show()

## 7. Choose the alert threshold on validation data

We try every cut-off on the validation precision-recall curve and keep the one with the best **F1**.
The test set is not touched here — otherwise the final results would be optimistic.

In [ ]:
val_scores = scores_for(X_val)
precision, recall, thresholds = precision_recall_curve(y_val, val_scores)

if len(thresholds):
    f1_values = 2 * precision[:-1] * recall[:-1] / (precision[:-1] + recall[:-1] + 1e-12)
    best_idx = int(np.argmax(f1_values))
    threshold = float(thresholds[best_idx])
else:
    threshold = 0.5

print(f"Selected threshold: {threshold:.6f}")
print(f"Validation precision={precision[best_idx]:.3f}  recall={recall[best_idx]:.3f}  F1={f1_values[best_idx]:.3f}")

plt.figure(figsize=(6, 4.5))
plt.plot(recall, precision, color="#1E293B")
plt.scatter(recall[best_idx], precision[best_idx], color="#EE4C2C", zorder=3, label=f"Best F1 @ {threshold:.4f}")
plt.xlabel("Recall")
plt.ylabel("Precision")
plt.title("Validation precision-recall curve")
plt.legend()
plt.show()

## 8. Final evaluation on the held-out test set

Used exactly once. Scores rank risk — because of the weighted loss they are **not calibrated probabilities**,
which is also why the threshold ends up very close to 1.

In [ ]:
test_scores = scores_for(X_test)
test_preds = (test_scores >= threshold).astype(int)

tn, fp, fn, tp = confusion_matrix(y_test, test_preds, labels=[0, 1]).ravel()
metrics = {
    "threshold": threshold,
    "average_precision_pr_auc": float(average_precision_score(y_test, test_scores)),
    "roc_auc": float(roc_auc_score(y_test, test_scores)),
    "confusion_matrix_tn_fp_fn_tp": [int(tn), int(fp), int(fn), int(tp)],
    "classification_report": classification_report(
        y_test, test_preds, labels=[0, 1], target_names=["legitimate", "fraud"],
        zero_division=0, output_dict=True,
    ),
}

print(f"PR-AUC / average precision: {metrics['average_precision_pr_auc']:.4f}")
print(f"ROC-AUC:                    {metrics['roc_auc']:.4f}")
print(f"Confusion matrix [TN, FP, FN, TP]: {metrics['confusion_matrix_tn_fp_fn_tp']}")
print(f"Fraud caught: {tp} of {tp + fn}   False alarms: {fp} of {tn + fp:,} legitimate\n")
print(classification_report(y_test, test_preds, labels=[0, 1], target_names=["legitimate", "fraud"], zero_division=0))

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(17, 4.5))

ConfusionMatrixDisplay(
    confusion_matrix(y_test, test_preds, labels=[0, 1]),
    display_labels=["Legitimate", "Fraud"],
).plot(ax=axes[0], cmap="Oranges", colorbar=False, values_format=",")
axes[0].set_title("Test confusion matrix")

fpr, tpr, _ = roc_curve(y_test, test_scores)
axes[1].plot(fpr, tpr, color="#0F766E", label=f"ROC-AUC = {metrics['roc_auc']:.3f}")
axes[1].plot([0, 1], [0, 1], "--", color="#CBD5E1")
axes[1].set_xlabel("False positive rate")
axes[1].set_ylabel("True positive rate")
axes[1].set_title("Test ROC curve")
axes[1].legend()

axes[2].hist(test_scores[y_test == 0], bins=50, alpha=0.6, density=True, label="Legitimate", color="#0F766E")
axes[2].hist(test_scores[y_test == 1], bins=50, alpha=0.6, density=True, label="Fraud", color="#EE4C2C")
axes[2].axvline(threshold, color="#1E293B", linestyle="--", label="Threshold")
axes[2].set_yscale("log")
axes[2].set_xlabel("Risk score")
axes[2].set_title("Test score distribution")
axes[2].legend()

plt.tight_layout()
plt.show()

## 9. Score individual transactions

This is the same logic as the `/predict` endpoint in `app/api.py`: scale with the saved scaler → FraudNet → sigmoid → compare with the threshold.

In [ ]:
def predict_transaction(features: dict) -> dict:
    missing = sorted(set(feature_columns) - set(features))
    extra = sorted(set(features) - set(feature_columns))
    if missing or extra:
        raise ValueError({"missing_features": missing, "extra_features": extra})

    values = np.asarray([float(features[c]) for c in feature_columns], dtype=np.float64)
    transformed = scaler.transform(values.reshape(1, -1)).astype("float32")

    model.eval()
    with torch.no_grad():
        score = float(torch.sigmoid(model(torch.from_numpy(transformed).to(device))).item())

    flagged = score >= threshold
    return {
        "risk_score": round(score, 6),
        "threshold": round(threshold, 6),
        "decision": "REVIEW" if flagged else "LOWER_RISK",
        "flagged_for_review": flagged,
    }


# Demo rows (from the full dataset, so some were seen during training).
for row_idx in [0, 541, 72757, 460]:
    row = df.iloc[row_idx]
    result = predict_transaction({c: row[c] for c in feature_columns})
    actual = "Fraud" if row["Class"] == 1 else "Legitimate"
    print(f"Row {row_idx:>6} | actual={actual:<10} | score={result['risk_score']:.6f} | {result['decision']}")

## 10. Save the artifacts

These three files are what `app/api.py` loads. Download them from the Kaggle *Output* tab and copy them into the project's `artifacts/` folder to serve this model locally.

In [ ]:
torch.save(
    {"model_state_dict": model.cpu().state_dict(), "input_size": len(feature_columns)},
    ARTIFACT_DIR / "fraud_model.pt",
)
model.to(device)
joblib.dump(scaler, ARTIFACT_DIR / "scaler.joblib")

metadata = {
    "feature_columns": feature_columns,
    "threshold": threshold,
    "metrics": metrics,
    "seed": SEED,
    "epochs": EPOCHS,
}
with open(ARTIFACT_DIR / "metadata.json", "w") as f:
    json.dump(metadata, f, indent=2)

print("Saved to", ARTIFACT_DIR.resolve())
for p in sorted(ARTIFACT_DIR.iterdir()):
    print(f"  {p.name:<16} {p.stat().st_size:>8,} bytes")

## Summary and limitations

- A small neural network (~4,100 weights) with a weighted loss detects most fraud in a dataset where fraud is only 0.17%.
- The threshold is tuned on validation data; the test set is used once.
- **Limitations:** historical anonymized data (2013), uncalibrated scores, fixed hyperparameters and a single run, only 74 test frauds.
- Results can differ slightly between CPU, Apple MPS and Kaggle GPU because floating-point operations differ.

**Next steps:** hyperparameter tuning with early stopping, comparison with logistic regression and XGBoost, and SHAP explanations.